# PoG Coordinate Origin Investigation

**Question:** Is `(0,0)` at the camera or at the screen center?

**Why it matters:** The EDA labeling function `label_pog_5` treats `(0,0)` as the neutral center. If the origin is at the camera (top of device), the entire labeling system is shifted — "Straight" means looking at the camera, "Down" absorbs center-of-screen gazes, and "Up" only captures gazes above the camera.

**What we already know from the GazeCapture repo:**
> `XCam, YCam`: Position of the dot measured in **centimeters relative to the camera center**. `YCam` values will be **negative for portrait mode** since the screen is below the camera.

This notebook confirms that empirically with our data.

In [ ]:
# ============================================================
# Setup
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
# ============================================================
# !!! UPDATE THIS PATH
# !!! This is the 849K-row labels CSV saved by the EDA notebook to Drive.
# !!! Copy it into data_MIT/ or point this path to wherever it lives.
# !!! It should have columns: subject_id, frame_idx, pog_x, pog_y, label
# ============================================================

LABELS_PATH = './pog_labels.csv'

df = pd.read_csv(LABELS_PATH)
print(f"Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df[['pog_x', 'pog_y']].describe()

## 1. Where is the density?

If `(0,0)` is at the **camera** (top of device in portrait mode), the densest cluster should sit well below zero on Y — roughly where the screen center would be (~6–7 cm below camera on a typical iPhone).

If `(0,0)` were at **screen center**, density would cluster near `(0, 0)`.

In [ ]:
# ============================================================
# 2D density plot + marginal Y distribution
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: 2D heatmap ---
h = axes[0].hist2d(
    df['pog_x'], df['pog_y'],
    bins=100,
    range=[[-15, 15], [-15, 15]],
    cmap='inferno'
)
axes[0].axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.7)
axes[0].axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.7)
axes[0].set_xlabel('pog_x (cm from camera)')
axes[0].set_ylabel('pog_y (cm from camera)')
axes[0].set_title('PoG Density — Where is the mass?')
axes[0].set_aspect('equal')
plt.colorbar(h[3], ax=axes[0], label='frame count')

# --- Right: marginal Y histogram ---
axes[1].hist(df['pog_y'], bins=200, color='teal', alpha=0.7, edgecolor='none')
axes[1].axvline(0, color='red', linewidth=1.5, linestyle='--', label='y = 0 (camera)')
axes[1].set_xlabel('pog_y (cm from camera)')
axes[1].set_ylabel('frame count')
axes[1].set_title('pog_y Marginal — Negative = Below Camera')
axes[1].legend()

plt.tight_layout()
plt.show()

# --- Summary stats ---
median_y = df['pog_y'].median()
median_x = df['pog_x'].median()
print(f"\nMedian pog_x: {median_x:.2f} cm")
print(f"Median pog_y: {median_y:.2f} cm")
print(f"\nExpected if camera-origin: median_y ≈ -6 to -7 cm (screen center on a typical iPhone)")
print(f"Your median_y = {median_y:.2f}")

## 2. Label boundaries overlaid on the data

This shows where the current `label_pog_5` decision boundaries fall relative to where people are actually looking. The Straight box is drawn at `(0,0)` = camera. If the data mass is offset below that, it confirms the labeling is centered on the wrong point.

In [ ]:
# ============================================================
# Scatter with label coloring + decision boundaries
# ============================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Subsample for plotting speed
sample = df.sample(n=min(50_000, len(df)), random_state=42)

# Color by current label
label_colors = {
    'Up':       '#e74c3c',
    'Down':     '#3498db',
    'Left':     '#2ecc71',
    'Right':    '#f39c12',
    'Straight': '#95a5a6'
}

for label, color in label_colors.items():
    mask = sample['label'] == label
    if mask.sum() > 0:
        ax.scatter(
            sample.loc[mask, 'pog_x'],
            sample.loc[mask, 'pog_y'],
            c=color, s=1, alpha=0.3, label=f"{label} ({mask.sum():,})"
        )

# Current Straight box: centered at (0,0) = camera, threshold=4
rect = Rectangle((-4, -4), 8, 8, linewidth=2,
                 edgecolor='gray', facecolor='none',
                 linestyle='--', label='Straight zone (at camera)')
ax.add_patch(rect)

# Diagonal decision boundaries: y=x and y=-x
lim = 20
ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.5, alpha=0.5)
ax.plot([-lim, lim], [lim, -lim], 'k--', linewidth=0.5, alpha=0.5)

# Mark camera origin
ax.plot(0, 0, 'k+', markersize=15, markeredgewidth=2, label='(0,0) = Camera')

# Mark approximate screen center
# !!! Typical iPhone portrait: camera is ~6-7 cm above screen center
APPROX_SCREEN_CENTER_Y = -6.5
ax.plot(0, APPROX_SCREEN_CENTER_Y, 'r+', markersize=15,
        markeredgewidth=2, label=f'≈ Screen center (0, {APPROX_SCREEN_CENTER_Y})')

ax.set_xlim(-15, 15)
ax.set_ylim(-15, 15)
ax.set_xlabel('pog_x (cm from camera)')
ax.set_ylabel('pog_y (cm from camera)')
ax.set_title('Current Labels with Camera-Origin Decision Boundaries')
ax.set_aspect('equal')
ax.legend(markerscale=5, loc='upper left')

plt.tight_layout()
plt.show()

## 3. How much does it actually affect the 4-class model?

For the 4-class model (no Straight), the labeling uses `abs(x) > abs(y)` centered at the camera. Recentering to the data centroid would shift the diagonal boundaries. This cell quantifies how many samples would change labels.

In [ ]:
# ============================================================
# Quantify label changes if we recenter to data centroid
# ============================================================

# Compute the centroid of all gaze points (≈ screen center)
centroid_x = df['pog_x'].median()
centroid_y = df['pog_y'].median()
print(f"Data centroid: ({centroid_x:.2f}, {centroid_y:.2f})")
print(f"This is our best empirical estimate of 'screen center' in camera coords.\n")

# Shift coordinates so centroid becomes (0,0)
df['pog_x_shifted'] = df['pog_x'] - centroid_x
df['pog_y_shifted'] = df['pog_y'] - centroid_y


# Re-label with the 4-class scheme on shifted coordinates
def label_4class(x, y):
    """Same logic as label_pog_5 but without Straight."""
    if abs(x) > abs(y):
        return 'Left' if x < 0 else 'Right'
    else:
        return 'Down' if y < 0 else 'Up'


# Current 4-class labels (camera origin, no Straight zone)
df['label_4c_camera'] = df.apply(
    lambda r: label_4class(r['pog_x'], r['pog_y']), axis=1
)

# Recentered 4-class labels
df['label_4c_shifted'] = df.apply(
    lambda r: label_4class(r['pog_x_shifted'], r['pog_y_shifted']), axis=1
)

# Compare
changed = df['label_4c_camera'] != df['label_4c_shifted']
print(f"Total samples: {len(df):,}")
print(f"Labels that would change: {changed.sum():,} ({changed.mean()*100:.1f}%)\n")

# Distribution comparison
print("--- Camera-origin labels ---")
cam_counts = df['label_4c_camera'].value_counts()
for label in ['Up', 'Down', 'Left', 'Right']:
    ct = cam_counts.get(label, 0)
    print(f"  {label:6s}: {ct:>7,}  ({ct/len(df)*100:.1f}%)")

print("\n--- Centroid-recentered labels ---")
shift_counts = df['label_4c_shifted'].value_counts()
for label in ['Up', 'Down', 'Left', 'Right']:
    ct = shift_counts.get(label, 0)
    print(f"  {label:6s}: {ct:>7,}  ({ct/len(df)*100:.1f}%)")

In [ ]:
# ============================================================
# Side-by-side: camera-origin vs recentered boundaries
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sample = df.sample(n=min(50_000, len(df)), random_state=42)

for ax, x_col, y_col, title_str in [
    (axes[0], 'pog_x', 'pog_y', 'Camera-origin (current)'),
    (axes[1], 'pog_x_shifted', 'pog_y_shifted', 'Recentered to data centroid')
]:
    # Which label column to use for coloring
    lbl_col = 'label_4c_camera' if 'shifted' not in x_col else 'label_4c_shifted'

    for label, color in [('Up','#e74c3c'), ('Down','#3498db'),
                          ('Left','#2ecc71'), ('Right','#f39c12')]:
        mask = sample[lbl_col] == label
        if mask.sum() > 0:
            ax.scatter(
                sample.loc[mask, x_col],
                sample.loc[mask, y_col],
                c=color, s=1, alpha=0.3, label=label
            )

    # Decision boundaries at this coordinate system's origin
    ax.plot([-20, 20], [-20, 20], 'k--', linewidth=0.5, alpha=0.5)
    ax.plot([-20, 20], [20, -20], 'k--', linewidth=0.5, alpha=0.5)
    ax.axhline(0, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(0, color='gray', linewidth=0.5, alpha=0.3)
    ax.plot(0, 0, 'k+', markersize=12, markeredgewidth=2)

    ax.set_xlim(-15, 15)
    ax.set_ylim(-15, 15)
    ax.set_aspect('equal')
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.set_title(title_str)
    ax.legend(markerscale=5, loc='upper left')

plt.tight_layout()
plt.show()

## Takeaways

**Document your findings here after running:**

- Median pog_y = ?
- % of labels that change with recentering = ?
- Up count camera-origin vs recentered = ?

**Key question for the team:** Does the camera-origin labeling hurt the 4-class model's real-world utility, or does the model learn the correct image→direction mapping regardless of where we draw the coordinate boundary? The answer depends on whether we care about *screen-relative* directions or *camera-relative* directions for the assistive UI.